# 07 — What the ASR is plotted against

Who pays for the emissions in notebook 04, and who was paid for them.

The ASR says whether a country lives inside its share of the carbon budget. It
says nothing about what happens to that country when the budget is overshot,
who did the overshooting, or who could afford to stop. This notebook joins six
candidate **x axes** onto the ASR panel, so the Pacific can be read against the
world under each of them in the same scatter.

| axis | field | the question it puts on the x axis |
|---|---|---|
| ND-GAIN exposure | `exposure` | who stands in the way of the harm |
| Disaster loss, % of GDP | `loss` | who is already paying for it |
| Share of world emissions | `share` | who caused it |
| Fossil-fuel rents, % of GDP | `rents` | who was paid for causing it |
| GDP per capita, PPP | `gdp` | who can afford to act |
| ND-GAIN vulnerability | `vuln` | the axis everyone else plots, kept as a counter-example |

## The axis that cannot be used

Cumulative emissions **per capita** is the obvious candidate and it is
unusable: it correlates **+1.00** with the egalitarian ASR, because that ASR
*is* per-capita emissions divided by a per-capita fair share. Plotting it would
draw a straight line and call it a finding. Share of world emissions is the
usable form of the same question — population-weighted rather than per-capita —
and lands at +0.43. Section 6 reproduces every one of these correlations.

## Why ND-GAIN and not INFORM or the WorldRiskIndex

Most global risk indices score *absolute expected humanitarian impact*, which
is population-weighted. A country of 11,000 people cannot rank high on them no
matter what happens to it: INFORM Risk 2026 puts Tuvalu 180th of 191 on hazard,
and the WorldRiskIndex 2025 puts it 164th of 193. Both would draw the Pacific as
safe, which is the opposite of the argument here and would be wrong.

ND-GAIN is structural rather than population-weighted, so smallness does not
read as safety.

## Why exposure and not ND-GAIN's headline vulnerability score

ND-GAIN vulnerability is the mean of three components — exposure, sensitivity
and adaptive capacity. The last two are largely development indicators, and it
shows: across the 183 countries with both numbers at 2023, the headline
vulnerability score correlates **-0.83** with log GDP per capita, adaptive
capacity **-0.86**, sensitivity **-0.66**. A chart built on the composite is
open to the fair reply that it plots poverty and calls it climate.

**Exposure** is the physical component alone — projected change in sea level,
temperature, precipitation, cereal yield, marine biodiversity, and the share of
population and infrastructure in the way of it. It correlates **-0.50** with
GDP per capita: still not independent of wealth, because poor countries really
do sit in worse places, but far from a restatement of it. On exposure Tuvalu
ranks 2nd of 192 countries behind the Maldives, and seven Pacific islands sit in
the global top 20.

The composite ships in the panel too, as `vuln`. It is not a rival axis but the
exhibit for this argument: the chart most entries draw, offered next to the one
this project draws, with the correlations attached.

## Where each axis comes from, and why not all of them come from SPC

Every axis was checked against the Pacific Data Hub first. SPC is the right
first stop and the wrong last one, for a structural reason: an axis that places
the Pacific *against the world* needs the world on it, and SPC publishes only
the PICTs.

- **Exposure** — no SPC equivalent at all, across all 127 dataflows. The
  nearest thing, `DF_POP_LECZ` (share of population in the low-elevation
  coastal zone), is Pacific-only. → ND-GAIN.
- **Disaster loss** — SPC *does* carry it: `DF_SDG_11`, series `VC_DSR_LSGP`,
  SDG indicator 11.5.2, for 12 PICTs. It is the same Sendai Framework
  reporting that reaches the UN SDG Global Database, so the world axis is
  pulled from the UN and checked against SPC row by row in section 2.
- **Fossil-fuel rents** — nothing in the SPC catalogue reports resource rents:
  no match across the 127 dataflow names or the 860 SDG series. → World Bank.
- **Share of world emissions** — computed in notebook 04, on the SPC emissions
  this project already uses for the Pacific.

In [1]:
import io
import json
import zipfile

import numpy as np
import pandas as pd
import requests

from config import PACIFIC, PACIFIC_ISO3, ROOT, VIZ, YEARS
from pdh_api import fetch_data_pacific

YEAR = max(YEARS)  # 2023
Y = str(YEAR)

# ND-GAIN publishes the Country Index as one zip of CSVs, updated annually.
# The 2026 edition carries 1995-2024. The asset id changes with each edition:
# check https://gain.nd.edu/our-work/country-index/download-data/ if this 404s.
NDGAIN_URL = "https://gain.nd.edu/assets/647440/ndgain_countryindex_2026.zip"
NDGAIN_DIR = ROOT / "data_raw" / "ndgain"

# Both live under vulnerability/ in the archive — exposure is a component of
# the composite, not a separate index.
WANTED = ["vulnerability/exposure.csv", "vulnerability/vulnerability.csv"]
EXPOSURE_CSV = NDGAIN_DIR / "exposure.csv"
VULN_CSV = NDGAIN_DIR / "vulnerability.csv"

# Direct economic loss attributed to disasters, relative to GDP: SDG indicator
# 11.5.2, reported by countries under the Sendai Framework. The UN SDG Global
# Database is the world-wide copy of the series SPC republishes for the PICTs.
SDG_API = "https://unstats.un.org/sdgapi/v1/sdg/Series/Data"
LOSS_SERIES = "VC_DSR_LSGP"
LOSS_YEARS = range(2015, 2025)  # the Sendai Framework reporting period
LOSS_CSV = ROOT / "data_raw" / "disaster_loss" / "sdg_11_5_2.csv"

# Fossil-fuel rents: the three fossil components of the World Bank's natural
# resource rents, summed. Rent is the value of extraction above its cost — the
# part of a country's GDP that is paid for selling the carbon, so it is the
# receipts side of the same overshoot the ASR measures.
WB_API = "https://api.worldbank.org/v2/country/all/indicator"
WB_RENTS = {
    "NY.GDP.PETR.RT.ZS": "oil",
    "NY.GDP.COAL.RT.ZS": "coal",
    "NY.GDP.NGAS.RT.ZS": "gas",
}
# 2021 is the last year the World Bank publishes the rents series, and the mean
# over the window is used rather than one year, because rents track the oil
# price: 2020 alone would halve every petrostate.
RENTS_YEARS = range(2015, 2022)
RENTS_CSV = ROOT / "data_raw" / "fossil_rents" / "wb_fossil_rents.csv"

## 1. Download

The zip is ~4.7 MB and holds every ND-GAIN indicator; we keep two tables. The
server rejects requests without a browser user agent — it answers a bare
request with a 403, not the file.


In [2]:
if not all((NDGAIN_DIR / f.split("/")[-1]).exists() for f in WANTED):
    NDGAIN_DIR.mkdir(parents=True, exist_ok=True)
    resp = requests.get(
        NDGAIN_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=300
    )
    resp.raise_for_status()
    print(f"downloaded {len(resp.content) / 1e6:.1f} MB")

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        # The archive's top folder has been named both 'resources' and
        # 'resources 2' across editions, so match on the tail of the path.
        for wanted in WANTED:
            members = [
                n for n in zf.namelist()
                if n.endswith(wanted) and not n.startswith("__MACOSX")
            ]
            assert len(members) == 1, f"expected one {wanted}, found {members}"
            (NDGAIN_DIR / wanted.split("/")[-1]).write_bytes(zf.read(members[0]))

print(EXPOSURE_CSV)


/Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/data_raw/ndgain/exposure.csv


## 1b. Download — disaster loss and fossil rents

Two APIs, both open and keyless, both cached to `data_raw/` so a rerun is
offline. Neither returns ISO3 the same way: the UN keys on M49 numeric codes,
which `countries.csv` from notebook 04 carries as `iso_n3`, while the World
Bank ships ISO3 already.

In [3]:
def fetch_sdg_series(code):
    """Every observation of one SDG series, all countries and years."""
    rows, page = [], 1
    while True:
        resp = requests.get(
            SDG_API, params={"seriesCode": code, "pageSize": 1000, "page": page}, timeout=180
        )
        resp.raise_for_status()
        payload = resp.json()
        rows += payload["data"]
        if page >= payload["totalPages"]:
            break
        page += 1
    out = pd.DataFrame(rows)[["geoAreaCode", "geoAreaName", "timePeriodStart", "value"]]
    return out.rename(columns={"timePeriodStart": "year"})


def fetch_wb_indicator(code, years):
    """One World Bank indicator over a year range, long."""
    rows, page = [], 1
    while True:
        resp = requests.get(
            f"{WB_API}/{code}",
            params={
                "date": f"{min(years)}:{max(years)}",
                "format": "json",
                "per_page": 10000,
                "page": page,
            },
            timeout=180,
        )
        resp.raise_for_status()
        meta, batch = resp.json()
        rows += batch
        if page >= meta["pages"]:
            break
        page += 1
    return pd.DataFrame(
        [{"iso3": r["countryiso3code"], "year": int(r["date"]), "value": r["value"]} for r in rows]
    )


if not LOSS_CSV.exists():
    LOSS_CSV.parent.mkdir(parents=True, exist_ok=True)
    fetch_sdg_series(LOSS_SERIES).to_csv(LOSS_CSV, index=False)

if not RENTS_CSV.exists():
    RENTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(
        [fetch_wb_indicator(code, RENTS_YEARS).assign(fuel=fuel) for code, fuel in WB_RENTS.items()]
    ).to_csv(RENTS_CSV, index=False)

loss_raw = pd.read_csv(LOSS_CSV)
rents_raw = pd.read_csv(RENTS_CSV)
print(f"disaster loss: {len(loss_raw)} observations, {loss_raw['geoAreaCode'].nunique()} areas")
print(f"fossil rents:  {len(rents_raw)} observations, {rents_raw['iso3'].nunique()} areas")

disaster loss: 3538 observations, 163 areas
fossil rents:  5565 observations, 260 areas


## 2. The SPC cross-check

The Pacific Data Hub publishes SDG 11.5.2 for the PICTs that report it. This is
the same Sendai reporting the UN aggregates, so the two should agree exactly —
and if they do, the world axis pulled from the UN is the SPC series with the
rest of the world attached, which is what lets the Pacific points be read
against the world ones.

The assertion below is that check. SPC rounds the series to two decimals and
the UN carries full precision, so the comparison is against the rounded UN
value; the panel itself keeps the unrounded numbers.

In [4]:
# DF_SDG_11 has 15 dimensions and the key needs a position for every one of
# them, so the trailing dots are not optional. Only FREQ and SERIES are pinned.
key = ".".join(["A", "", LOSS_SERIES] + [""] * 12)
spc_loss = fetch_data_pacific("DF_SDG_11", str(min(LOSS_YEARS)), str(max(LOSS_YEARS)), key, v="4.4")
spc_loss = spc_loss.assign(
    iso3=spc_loss["REF_AREA"].map(PACIFIC), year=spc_loss["TIME_PERIOD"].astype(int)
)[["iso3", "year", "value"]]
print(f"SPC reports {len(spc_loss)} country-years across {spc_loss['iso3'].nunique()} PICTs")

countries = pd.read_csv(VIZ / "countries.csv").set_index("iso_code")
m49 = pd.read_csv(VIZ / "contributions.csv").set_index("iso_n3")["iso_code"]

un_loss = loss_raw.assign(iso3=loss_raw["geoAreaCode"].map(m49))
# The API returns every observation twice, once per attribute variant.
un_loss = un_loss.dropna(subset=["iso3"])[["iso3", "year", "value"]].drop_duplicates()

both = spc_loss.merge(un_loss, on=["iso3", "year"], suffixes=("_spc", "_un"))
gap = (both["value_spc"] - both["value_un"].round(2)).abs().max()
print(f"{len(both)} country-years in both; largest disagreement {gap:.2g} pp")
assert gap < 1e-9, both[(both["value_spc"] - both["value_un"].round(2)).abs() >= 1e-9]
print("SPC and the UN publish the same numbers — the Pacific end of this axis is SPC data")

SPC reports 47 country-years across 12 PICTs
47 country-years in both; largest disagreement 0 pp
SPC and the UN publish the same numbers — the Pacific end of this axis is SPC data


## 3. Join

Every axis meets the ASR on ISO3, at or ending in one year:

- **ASR** — the three allocation rules from notebook 03, `{iso: {year: ratio}}`
- **ND-GAIN exposure and vulnerability** — wide, one column per year 1995-2024
- **GDP per capita** — World Bank PPP in constant 2017 USD, downloaded by
  pyaesa in notebook 01; the same GDP the prioritarian rule is built on, so
  the alternative x axis and the `pr` series are on one definition.
- **Share of world emissions** — notebook 04's cumulative 2000-2023 total, as a
  percentage of the world's
- **Disaster loss** — mean of the reported years 2015-2024, as a percentage of
  GDP. A reported zero is a real observation (a year with no qualifying loss)
  and is kept; an unreported year is dropped rather than read as zero, which
  would reward not reporting.
- **Fossil rents** — oil plus coal plus gas rents, mean 2015-2021, as a
  percentage of GDP

In [5]:
asr = {
    "eg": json.load(open(VIZ / "asr.json")),
    "gf": json.load(open(VIZ / "asr_gf.json")),
    "pr": json.load(open(VIZ / "asr_gdp.json")),
}

exposure = pd.read_csv(EXPOSURE_CSV).set_index("ISO3")[Y].dropna()
vulnerability = pd.read_csv(VULN_CSV).set_index("ISO3")[Y].dropna()

wb = pd.read_csv(ROOT / "data_processed" / "pop_gdp" / "wb_processed.csv")
wb_gdp = wb[wb["variable"] == "GDP|PPP"].set_index("iso3_code")[Y]
wb_pop = wb[wb["variable"] == "Population"].set_index("iso3_code")[Y]
gdp_pc = (wb_gdp / wb_pop).dropna()

share = pd.read_csv(VIZ / "contributions.csv").set_index("iso_code")["share_pct"]

loss = un_loss[un_loss["year"].isin(LOSS_YEARS)].groupby("iso3")["value"].mean().dropna()

rents = (
    rents_raw[rents_raw["year"].isin(RENTS_YEARS)]
    .pivot_table(index=["iso3", "year"], columns="fuel", values="value")
    .sum(axis=1, min_count=1)  # a country-year missing all three fuels stays null
    .groupby("iso3")
    .mean()
    .dropna()
)

for name, series in [
    ("ND-GAIN exposure", exposure),
    ("ND-GAIN vulnerability", vulnerability),
    ("GDP per capita", gdp_pc),
    ("share of world emissions", share),
    ("disaster loss / GDP", loss),
    ("fossil rents / GDP", rents),
]:
    print(f"{len(series):>4} countries with {name}")
print(f"{len(asr['eg']):>4} countries with an ASR")

 192 countries with ND-GAIN exposure
 190 countries with ND-GAIN vulnerability
 198 countries with GDP per capita
 198 countries with share of world emissions
 149 countries with disaster loss / GDP
 245 countries with fossil rents / GDP
 198 countries with an ASR


In [6]:
def value_of(series, iso, digits):
    """One country's number, or None where the source has no row for it. The
    scatter drops a point rather than inventing a position for it."""
    return round(float(series[iso]), digits) if iso in series.index else None


def row(iso):
    return {
        "iso": iso,
        "name": countries.loc[iso, "name"],
        "pacific": iso in PACIFIC_ISO3,
        "exposure": value_of(exposure, iso, 4),
        "vuln": value_of(vulnerability, iso, 4),
        "gdp": value_of(gdp_pc, iso, 1),
        "share": value_of(share, iso, 6),
        "loss": value_of(loss, iso, 5),
        "rents": value_of(rents, iso, 3),
        "asr": {
            rule: (round(table[iso][Y], 4) if iso in table and Y in table[iso] else None)
            for rule, table in asr.items()
        },
    }


X_AXES = ["exposure", "loss", "share", "rents", "gdp", "vuln"]
panel = [row(iso) for iso in sorted(asr["eg"])]
print(f"{len(panel)} countries in the panel")

198 countries in the panel


## 4. What the join drops

Two gaps, of different kinds.

The first is the same one throughout: an index of *countries* has no row for a
territory that is not a country. New Caledonia and French Polynesia have an ASR
— SPC reports their emissions and pyaesa allocates to them — but no ND-GAIN
score and no World Bank GDP, so they carry an ASR and no position on most axes.
This is the exclusion already noted for American Samoa, Guam and the Northern
Marianas in `config.py`, one step further along. `share` is the exception: it
comes from this project's own emissions table, so both territories sit on it.

The second is specific to disaster loss and belongs on the chart. 11.5.2 is
*reported* loss, and reporting is voluntary: about fifty countries in the panel
have never filed, and those that do file undercount — Vanuatu's largest
reported year is 0.41% of GDP over a period that includes Cyclone Pam. The axis
is a floor rather than a measurement, and its world cloud is the thinnest of
the six.

In [7]:
plotted = {
    axis: sum(1 for c in panel if c[axis] is not None and c["asr"]["eg"] is not None)
    for axis in X_AXES
}
for axis, n in sorted(plotted.items(), key=lambda kv: -kv[1]):
    print(f"{n:>4} of {len(panel)} plottable against {axis}")

missing = [c["iso"] for c in panel if c["exposure"] is None]
print(f"\nno exposure score ({len(missing)}): {', '.join(missing)}")

pacific_gaps = {
    axis: [c["iso"] for c in panel if c["pacific"] and c[axis] is None] for axis in X_AXES
}
for axis, gaps in pacific_gaps.items():
    print(f"Pacific gaps on {axis}: {', '.join(gaps) or 'none'}")
assert set(pacific_gaps["exposure"]) <= {"NCL", "PYF"}, pacific_gaps["exposure"]

 198 of 198 plottable against share
 189 of 198 plottable against exposure
 189 of 198 plottable against rents
 188 of 198 plottable against gdp
 188 of 198 plottable against vuln
 149 of 198 plottable against loss

no exposure score (9): ABW, HKG, MAC, NCL, PYF, SSD, TCA, TWN, VGB
Pacific gaps on exposure: NCL, PYF
Pacific gaps on loss: NCL, PYF
Pacific gaps on share: none
Pacific gaps on rents: MHL, PLW
Pacific gaps on gdp: NCL, PYF
Pacific gaps on vuln: NCL, PYF


## 5. Where the Pacific lands

The scene's whole claim, as a table: exposure high, loss already being paid,
emissions and fossil receipts near nothing, ASR at or under a fair share. World
rank is out of the 192 countries ND-GAIN scores, 1 = most exposed.

In [8]:
world_rank = exposure.rank(ascending=False, method="min").astype(int)

pacific = pd.DataFrame([c for c in panel if c["pacific"]])
pacific["asr_eg"] = pacific["asr"].map(lambda a: a["eg"])
pacific["world_rank"] = pacific["iso"].map(world_rank)

print(
    pacific[["iso", "name", "exposure", "world_rank", "loss", "share", "rents", "gdp", "asr_eg"]]
    .sort_values("exposure", ascending=False)
    .to_string(index=False)
)

sellers = pacific.loc[pacific["rents"].fillna(0) > 0.5, "iso"]
print(f"\nPacific countries earning any fossil rent: {', '.join(sellers) or 'none'}")

iso                 name  exposure  world_rank    loss    share  rents     gdp  asr_eg
TUV               Tuvalu    0.6314         2.0 0.11706 0.000009  0.000  5321.8  0.4723
KIR             Kiribati    0.6185         5.0 0.03892 0.000245  0.000  2875.5  0.9446
TON                Tonga    0.6006        10.0 0.00446 0.000653  0.000  6437.0  3.1879
FSM Micronesia (country)    0.5977        11.0 0.00719 0.000130  0.000  3492.7  0.5904
SLB      Solomon Islands    0.5907        15.0 0.01845 0.001294  0.000  2142.1  0.8265
MHL     Marshall Islands    0.5870        19.0 0.00477 0.000009    NaN  6227.5  0.1181
NRU                Nauru    0.5864        20.0 0.00027 0.000003  0.000 11328.2  0.1181
WSM                Samoa    0.5415        47.0 0.00118 0.001152  0.000  6760.5  2.7156
PLW                Palau    0.5317        56.0 0.03691 0.003380    NaN 15039.5 97.4092
PNG     Papua New Guinea    0.5296        58.0 0.00007 0.024015  8.239  3850.8  1.1807
VUT              Vanuatu    0.5216        6

## 6. What each axis is independent of

Two correlations decide whether an axis is worth drawing.

**Against log GDP per capita** — the objection that a vulnerability axis is a
wealth axis in disguise. This is why the chart plots exposure and keeps the
ND-GAIN composite only as the counter-example.

**Against the egalitarian ASR** — the y axis itself. An x axis that correlates
too well with y is not a second variable, it is the same one twice. Cumulative
emissions per capita fails outright at +1.00, because the egalitarian ASR is
that number divided by a constant fair share. It is computed here so the
failure is on the record rather than in a footnote.

In [9]:
frame = pd.DataFrame(panel).set_index("iso")
frame["asr_eg"] = frame["asr"].map(lambda a: a["eg"])

contributions = pd.read_csv(VIZ / "contributions.csv").set_index("iso_code")
frame["cum_per_capita"] = contributions["emissions_t_per_capita"]  # the rejected axis

LOGGED = {"gdp", "share", "cum_per_capita"}
log_gdp = np.log10(frame["gdp"])
log_asr = np.log10(frame["asr_eg"])

print(f"{'axis':<16}{'n':>5}{'r vs log GDP pc':>18}{'r vs log ASR':>15}")
for axis in X_AXES + ["cum_per_capita"]:
    column = np.log10(frame[axis].clip(lower=1e-9)) if axis in LOGGED else frame[axis]
    n = frame[[axis, "asr_eg"]].dropna().shape[0]
    r_gdp = "—" if axis == "gdp" else f"{column.corr(log_gdp):+.2f}"
    print(f"{axis:<16}{n:>5}{r_gdp:>18}{column.corr(log_asr):>+15.2f}")

print("\nND-GAIN components against log GDP per capita, the reason for exposure:")
components = pd.DataFrame(
    {"vulnerability": vulnerability, "exposure": exposure, "gdp": gdp_pc}
).dropna()
for column in ("vulnerability", "exposure"):
    r = components[column].corr(np.log10(components["gdp"]))
    print(f"  {column:14s} r = {r:+.2f}  (n={len(components)})")

axis                n   r vs log GDP pc   r vs log ASR
exposure          189             -0.50          -0.40
loss              149             -0.33          -0.22
share             198             +0.31          +0.43
rents             189             +0.07          +0.28
gdp               188                 —          +0.82
vuln              188             -0.83          -0.67
cum_per_capita    198             +0.82          +1.00

ND-GAIN components against log GDP per capita, the reason for exposure:
  vulnerability  r = -0.83  (n=184)
  exposure       r = -0.50  (n=184)


## 7. Export

`scatter.json` is the fifth table the app fetches at runtime. The year is
carried in the file so the scene can label itself without hard-coding 2023 in
two places, and every axis carries its own source string, because the scene
prints the source of whichever axis is showing.

In [10]:
out = {
    "year": YEAR,
    "source": {
        "exposure": "ND-GAIN Country Index 2026, vulnerability/exposure (University of Notre Dame)",
        "vuln": "ND-GAIN Country Index 2026, vulnerability (University of Notre Dame)",
        "gdp": "World Bank GDP per capita, PPP, constant 2017 USD",
        "share": "Share of cumulative CO2 2000-2023, notebook 04",
        "loss": (
            "SDG 11.5.2, direct disaster loss relative to GDP, mean 2015-2024 "
            "(UN SDG Global Database; SPC DF_SDG_11 VC_DSR_LSGP for the Pacific)"
        ),
        "rents": "World Bank oil, coal and gas rents as % of GDP, mean 2015-2021",
        "asr": "pyaesa, notebook 03",
    },
    "countries": panel,
}

path = VIZ / "scatter.json"
path.write_text(json.dumps(out, separators=(",", ":")))
print(f"wrote {path} — {path.stat().st_size / 1024:.0f} KB")

wrote /Users/laurendurivault/Documents/GitHub/laurendudu/pacific-dataviz-challenge-2026/data_viz/scatter.json — 36 KB
